In [1]:
import os

In [2]:
%pwd

'c:\\Users\\abhin\\Desktop\\Mlops\\Deep-Learning-Kidney-Tumor-Classification-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\abhin\\Desktop\\Mlops\\Deep-Learning-Kidney-Tumor-Classification-'

In [16]:
import dagshub
dagshub.init(repo_owner='Abhitar3', repo_name='Deep-Learning-Kidney-Tumor-Classification-', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Initialized MLflow to track repo "Abhitar3/Deep-Learning-Kidney-Tumor-Classification-"

Repository Abhitar3/Deep-Learning-Kidney-Tumor-Classification- initialized!

🏃 View run crawling-shad-855 at: https://dagshub.com/Abhitar3/Deep-Learning-Kidney-Tumor-Classification-.mlflow/#/experiments/0/runs/39489f4bc7d74a63b8a3bd4ceffdf70b
🧪 View experiment at: https://dagshub.com/Abhitar3/Deep-Learning-Kidney-Tumor-Classification-.mlflow/#/experiments/0


In [17]:
import tensorflow as tf

In [18]:
model=tf.keras.models.load_model("artifacts/training/trained_model.h5")

In [19]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [20]:
from cnnclassifier.constants import *
from cnnclassifier.utils.common import read_yaml,create_directories,save_json

In [21]:
class configuartionManager:
    def __init__(self, config_file_path = config_file_path, params_file_path = params_file_path):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model=Path("artifacts/training/trained_model.h5"),
            training_data=Path("artifacts/data_ingestion/unzip/KidneyData"),
            mlflow_uri="https://dagshub.com/Abhitar3/Deep-Learning-Kidney-Tumor-Classification-.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )

        return eval_config

In [22]:
import tensorflow as tf
from pathlib import Path
from urllib.parse import urlparse
import mlflow
import mlflow.keras

# from cnnclassifier.entity.config_entity import EvaluationConfig
from cnnclassifier.utils.common import save_json


class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1.0 / 255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {
            "loss": self.score[0],
            "accuracy": self.score[1]
        }
        save_json(path=Path("scores.json"), data=scores)
    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)

        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)

            mlflow.log_metrics(
                {
                    "loss": self.score[0],
                    "accuracy": self.score[1]
                }
            )

            if tracking_url_type_store != "file":
                mlflow.keras.log_model(
                    self.model,
                    "model",
                    registered_model_name="VGG16Model"
                )
            else:
                mlflow.keras.log_model(
                    self.model,
                    "model"
                )

    

In [23]:
try:
    config = configuartionManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
    raise e

Found 2207 images belonging to 2 classes.
69/69 ━━━━━━━━━━━━━━━━━━━━ 368s 5s/step - accuracy: 0.7657 - loss: 3.6010


2026/08/06 13:52:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/06 13:52:04 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/08/06 13:52:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 2
Created version '2' of model 'VGG16Model'.


🏃 View run delicate-mink-120 at: https://dagshub.com/Abhitar3/Deep-Learning-Kidney-Tumor-Classification-.mlflow/#/experiments/0/runs/b145c3f521c64c068182ce71be726ad9
🧪 View experiment at: https://dagshub.com/Abhitar3/Deep-Learning-Kidney-Tumor-Classification-.mlflow/#/experiments/0
